# Brooklyn ambulance travel times

Build a directed driving-distance matrix from the proposed sites and existing firehouses to the same demand points: 5 sampled points per ZIP (`area_id`). Valhalla uses OpenStreetMap roads with its standard car access, one-way and turn rules. We request distance-based costing and disable hierarchy pruning. The route is between road positions selected by the router. The straight-line offsets to those positions are recorded separately.

The model uses **15 mph** for every route. We discard the router's travel-time estimate and calculate `minutes = 60 * network_miles / 15`. This is a driving-time proxy for a hypothetical standby-site study.

Run all cells from the repository root with a Python kernel and internet access. Both groups use the same Valhalla request settings, demand-point coordinates, and speed assumption.

| Input | How the notebook uses it |
|---|---|
| `data/candidate_sites.csv` | Reads the proposed sites' `site_id`, `lat`, and `lon`. |
| `data/FDNY_Firehouse_Listing_20261115.csv` | Keeps Brooklyn rows with coordinates and assigns `FH_001`, `FH_002`, etc. in file order, matching `Modelling_with_existing.ipynb`. |
| `data/demand_points_185.csv` | Reads `point_id`, `area_id`, `lat`, and `lon` for both sets of routes. Each `area_id` must have exactly 5 points. |

Each output row is one site → demand point route and carries both `point_id` and `area_id`, so the model can aggregate the 5 points of an area (e.g. mean or max minutes). The notebook validates both groups before writing `data/travel_times_routed_185.csv` for the candidates and `data/travel_times_existing_routed_185.csv` for the existing firehouses. With 20 candidates, 66 firehouses, and 37 areas × 5 points = 185 demand points, expect 3,700 and 12,210 rows.

In `Modelling_with_existing.ipynb`, point `TRAVEL_TIME_FILE` to `travel_times_routed_185.csv` and set `EXISTING_TIME_FILE = joinpath(DATA_DIR, "travel_times_existing_routed_185.csv")`. The firehouse file supplies the baseline locations used by that model.

Routing by [Valhalla](https://valhalla.github.io/valhalla/) via the FOSSGIS demo service. Map data © [OpenStreetMap contributors](https://www.openstreetmap.org/copyright). [Fix the map](https://www.openstreetmap.org/fixthemap).

In [8]:
import csv
import json
import math
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

DATA = Path("data")
CANDIDATE_FILE = DATA / "candidate_sites_v2.csv"
DEMAND_FILE = DATA / "demand_points_185.csv"
EXISTING_FILE = DATA / "FDNY_Firehouse_Listing_20261115.csv"
BOROUGH = "Brooklyn"
OUTPUT = DATA / "travel_times_routed_185.csv"
EXISTING_OUTPUT = DATA / "travel_times_existing_routed_185.csv"
MATRIX_URL = "https://valhalla1.openstreetmap.de/sources_to_targets"
SPEED_MPH = 15.0
METERS_PER_MILE = 1609.344
MAX_SNAP_METERS = 250
POINTS_PER_AREA = 5
MAX_PAIRS_PER_REQUEST = 100  # Public service limit: at most 100 pairs per request.
SITES_PER_REQUEST = 1
TARGETS_PER_REQUEST = MAX_PAIRS_PER_REQUEST // SITES_PER_REQUEST

def read_points(path, id_column, extra_columns=()):
    points = []
    seen_ids = set()
    with path.open(encoding="utf-8-sig", newline="") as source:
        reader = csv.DictReader(source)
        missing = {id_column, "lat", "lon", *extra_columns} - set(reader.fieldnames or ())
        if missing:
            raise ValueError(f"{path} is missing columns: {', '.join(sorted(missing))}")
        for row in reader:
            point_id = (row[id_column] or "").strip()
            if not point_id or point_id in seen_ids:
                raise ValueError(f"Missing or duplicate {id_column} on row {reader.line_num} of {path}")
            try:
                lat, lon = float(row["lat"]), float(row["lon"])
            except (TypeError, ValueError) as error:
                raise ValueError(f"Invalid coordinates for {point_id} in {path}") from error
            if not (-90 <= lat <= 90 and -180 <= lon <= 180):
                raise ValueError(f"Invalid coordinates for {point_id} in {path}")
            point = {"id": point_id, "lat": lat, "lon": lon}
            for column in extra_columns:
                value = (row[column] or "").strip()
                if not value:
                    raise ValueError(f"Missing {column} for {point_id} in {path}")
                point[column] = value
            points.append(point)
            seen_ids.add(point_id)
    if not points:
        raise ValueError(f"{path} has no input rows")
    return points


def read_existing_firehouses(path, borough):
    points = []
    with path.open(encoding="utf-8-sig", newline="") as source:
        reader = csv.DictReader(source)
        required = {"Borough", "Latitude", "Longitude"}
        missing = required - set(reader.fieldnames or ())
        if missing:
            raise ValueError(f"{path} is missing columns: {', '.join(sorted(missing))}")
        # Apply the same borough filter and missing-coordinate rule as the Julia model.
        # Number the surviving rows in input order so the two notebooks agree on IDs.
        for row in reader:
            if row["Borough"] != borough:
                continue
            lat_text = (row["Latitude"] or "").strip()
            lon_text = (row["Longitude"] or "").strip()
            if not lat_text or not lon_text:
                continue
            try:
                lat, lon = float(lat_text), float(lon_text)
            except ValueError as error:
                raise ValueError(f"Invalid firehouse coordinates on row {reader.line_num} of {path}") from error
            if not (-90 <= lat <= 90 and -180 <= lon <= 180):
                raise ValueError(f"Invalid firehouse coordinates on row {reader.line_num} of {path}")
            points.append({"id": f"FH_{len(points) + 1:03d}", "lat": lat, "lon": lon})
    if not points:
        raise ValueError(f"No {borough} firehouses with coordinates in {path}")
    return points

sites = read_points(CANDIDATE_FILE, "site_id")
existing_sites = read_existing_firehouses(EXISTING_FILE, BOROUGH)
# Several demand points per area: point_id is unique, area_id groups the points.
demand_points = read_points(DEMAND_FILE, "point_id", extra_columns=("area_id",))
points_per_area = {}
for point in demand_points:
    points_per_area[point["area_id"]] = points_per_area.get(point["area_id"], 0) + 1
bad_areas = {area: n for area, n in points_per_area.items() if n != POINTS_PER_AREA}
if bad_areas:
    raise ValueError(f"Expected {POINTS_PER_AREA} demand points per area_id, got: {bad_areas}")
candidate_ids = {site["id"] for site in sites}
existing_ids = {site["id"] for site in existing_sites}
overlap = candidate_ids & existing_ids
if overlap:
    raise ValueError(f"Candidate and existing site IDs overlap: {', '.join(sorted(overlap))}")
if OUTPUT.resolve() == EXISTING_OUTPUT.resolve():
    raise ValueError("Use different output files for the two groups.")
print(f"{len(sites)} candidates + {len(existing_sites)} firehouses × {len(demand_points)} demand points "
      f"({len(points_per_area)} areas × {POINTS_PER_AREA})")

20 candidates + 66 firehouses × 185 demand points (37 areas × 5)


In [9]:
def straight_line_meters(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(math.radians, (lat1, lon1, lat2, lon2))
    a = math.sin((lat2 - lat1) / 2)**2
    a += math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2)**2
    return 2 * 6_371_008.8 * math.asin(math.sqrt(min(1.0, a)))

def matrix_location(point):
    return {"lat": point["lat"], "lon": point["lon"], "search_cutoff": MAX_SNAP_METERS}

def request_matrix(batch, targets):
    payload = {
        "sources": [matrix_location(site) for site in batch],
        "targets": [matrix_location(point) for point in targets],
        "costing": "auto",
        "costing_options": {"auto": {
            "shortest": True,
            "disable_hierarchy_pruning": True,
        }},
        "units": "kilometers",
        "verbose": True,
    }
    request = Request(
        MATRIX_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "User-Agent": "15C57-Brooklyn-EMS-Study/1.0",
            "X-Client-Id": "15C57-Brooklyn-EMS-Study",
        },
    )
    try:
        with urlopen(request, timeout=60) as response:
            result = json.load(response)
    except HTTPError as error:
        detail = error.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Routing request failed ({error.code}): {detail}") from error
    except (URLError, TimeoutError) as error:
        raise RuntimeError(f"Could not reach the routing service: {error}") from error

    if result.get("warnings"):
        print("Router warnings:", result["warnings"])
    matrix = result.get("sources_to_targets")
    if result.get("units") != "kilometers" or not isinstance(matrix, list):
        raise ValueError("Expected a verbose distance matrix in kilometers.")
    if len(matrix) != len(batch) or any(len(row) != len(targets) for row in matrix):
        raise ValueError("Router returned an incomplete matrix.")
    return matrix

def route_record(site, point, route):
    pair = f"{site['id']} → {point['id']}"
    distance_km = route.get("distance")
    cost_meters = route.get("cost")
    if distance_km is None or not math.isfinite(distance_km) or distance_km < 0:
        raise ValueError(f"No valid driving distance for {pair}")
    # Distance-based auto costing should report cost in meters, within API rounding.
    if cost_meters is None or not math.isfinite(cost_meters) or abs(cost_meters - 1_000 * distance_km) > 1.1:
        raise ValueError(f"Router cost does not match distance-based costing for {pair}")

    site_snap = straight_line_meters(site["lat"], site["lon"], route["begin_lat"], route["begin_lon"])
    area_snap = straight_line_meters(point["lat"], point["lon"], route["end_lat"], route["end_lon"])
    if max(site_snap, area_snap) > MAX_SNAP_METERS + 1:
        raise ValueError(f"Road snap exceeds {MAX_SNAP_METERS} m for {pair}")

    route_miles = 1_000 * distance_km / METERS_PER_MILE
    return {
        "site_id": site["id"],
        "point_id": point["id"],
        "area_id": point["area_id"],
        "euclid_miles": straight_line_meters(site["lat"], site["lon"], point["lat"], point["lon"]) / METERS_PER_MILE,
        "network_miles": route_miles,
        "minutes": 60 * route_miles / SPEED_MPH,
        "site_snap_meters": site_snap,
        "area_snap_meters": area_snap,
    }

In [10]:
all_sites = sites + existing_sites
records = []
first_request = True
for offset in range(0, len(all_sites), SITES_PER_REQUEST):
    batch = all_sites[offset:offset + SITES_PER_REQUEST]
    # Split the demand points so each request stays within the pair limit.
    for target_offset in range(0, len(demand_points), TARGETS_PER_REQUEST):
        if not first_request:
            time.sleep(1.1)
        first_request = False
        targets = demand_points[target_offset:target_offset + TARGETS_PER_REQUEST]
        matrix = request_matrix(batch, targets)
        for source_index, row in enumerate(matrix):
            for target_index, route in enumerate(row):
                if (route.get("from_index"), route.get("to_index")) != (source_index, target_index):
                    raise ValueError("Router returned a route in an unexpected matrix position.")
                records.append(route_record(batch[source_index], targets[target_index], route))
    completed = offset + len(batch)
    if completed % 10 == 0 or completed == len(all_sites):
        print(f"Routed {completed} of {len(all_sites)} sites")

expected_pairs = {(site["id"], point["id"]) for site in all_sites for point in demand_points}
actual_pairs = {(row["site_id"], row["point_id"]) for row in records}
if len(records) != len(expected_pairs) or actual_pairs != expected_pairs:
    raise ValueError("Expected exactly one route per site and demand point.")

candidate_records = [row for row in records if row["site_id"] in candidate_ids]
existing_records = [row for row in records if row["site_id"] in existing_ids]
outputs = [(OUTPUT, candidate_records), (EXISTING_OUTPUT, existing_records)]

for output, rows in outputs:
    with output.with_suffix(output.suffix + ".tmp").open("w", encoding="utf-8", newline="") as destination:
        writer = csv.DictWriter(destination, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
for output, rows in outputs:
    output.with_suffix(output.suffix + ".tmp").replace(output)
    print(f"Wrote {len(rows):,} routes to {output}")
print(f"Generated at {datetime.now(timezone.utc).isoformat(timespec='seconds')}")

Routed 10 of 86 sites
Routed 20 of 86 sites
Routed 30 of 86 sites
Routed 40 of 86 sites
Routed 50 of 86 sites
Routed 60 of 86 sites
Routed 70 of 86 sites
Routed 80 of 86 sites
Routed 86 of 86 sites
Wrote 3,700 routes to data/travel_times_routed_185.csv
Wrote 12,210 routes to data/travel_times_existing_routed_185.csv
Generated at 2026-11-25T20:53:32+00:00
